# Hybrid COPV Verification and Abaqus Export

This notebook verifies the hybrid workflow implemented in this repository. It rebuilds the coarse COPV state, runs the hybrid winding + AFP optimizer, shows how the failure and manufacturing penalties influence the objective, visualizes Hashin and burst-pressure results on the geometry, visualizes the hybrid filament winding layout, and writes an Abaqus `.inp` bridge file for downstream validation.


## Setup

Reuse the packaged code from `src/copv_opt/`. This notebook is the main verification entrypoint for the repository's hybrid workflow.


In [ ]:
import os

ENABLE_X64 = False
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from jax import config as jax_config
jax_config.update("jax_enable_x64", ENABLE_X64)

import gc
import json
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display


FAILURE_MODE_NAMES = [
    "Fiber tension",
    "Fiber compression",
    "Matrix tension",
    "Matrix compression",
]


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing pyproject.toml and src/")


def hostify_tree(tree):
    return jax.tree_util.tree_map(
        lambda x: np.asarray(jax.device_get(x)) if isinstance(x, jax.Array) else x,
        tree,
    )


def binned_max_profile(s_coords, values, bins=36):
    s_coords = np.asarray(s_coords, dtype=np.float64).reshape(-1)
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    edges = np.linspace(float(np.min(s_coords)), float(np.max(s_coords)), bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    profile = np.full((bins,), np.nan, dtype=np.float64)
    for idx in range(bins):
        if idx == bins - 1:
            mask = (s_coords >= edges[idx]) & (s_coords <= edges[idx + 1])
        else:
            mask = (s_coords >= edges[idx]) & (s_coords < edges[idx + 1])
        if np.any(mask):
            profile[idx] = np.max(values[mask])
    return centers, profile


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs      : {OUTPUTS}")
print(f"JAX backend  : {jax.default_backend()}")
print(f"JAX devices  : {jax.devices()}")
print(f"JAX x64      : {ENABLE_X64}")

from copv_opt.abaqus_exporter import export_result_to_abaqus
from copv_opt.config import FailureConfig, FrictionConfig, GeometryConfig, HybridConfig, MaterialConfig, PatchConfig
from copv_opt.geometry import ensure_copv_mesh
from copv_opt.optimize import run_hybrid_optimization
from copv_opt.physics import baseline_response, build_copv_fem_state, estimate_burst_pressure_profile, make_solve_compliance
from copv_opt.visualize import (
    build_hybrid_winding_layout_data,
    build_patch_layout_data,
    plot_hybrid_winding_paths,
    plot_patch_projection,
    render_explicit_manufacturing_layout,
    render_vtu_interactive,
    save_explicit_manufacturing_layout_screenshot,
    save_layout_json,
    show_copv_mesh,
    write_vtu,
)


## COPV State Build

Build the same packaged COPV mesh/state used by the existing notebooks, then create the additional configs needed by the hybrid route.


In [ ]:
geom = GeometryConfig()
material = MaterialConfig()
failure_cfg = FailureConfig()
friction_cfg = FrictionConfig()
hybrid_cfg = HybridConfig()
verification_patch_cfg = PatchConfig(
    count=hybrid_cfg.patch_count,
    length=hybrid_cfg.patch_length,
    width=hybrid_cfg.patch_width,
)

step_path = OUTPUTS / "copv_shell.step"
msh_path = OUTPUTS / "copv_shell.msh"
mesh = ensure_copv_mesh(step_path, msh_path, geom, remesh=not msh_path.exists())
state = build_copv_fem_state(mesh.nodes, mesh.elems, material, geom)
solve_compliance = make_solve_compliance(state)
baseline = hostify_tree(baseline_response(state, material, solve_compliance))
gc.collect()

fig = show_copv_mesh(
    mesh.nodes,
    state["outer_faces"],
    geom,
    f"COPV analysis mesh: {len(mesh.nodes)} nodes / {len(mesh.elems)} tetra",
    OUTPUTS / "copv_analysis_mesh.png",
)
display(fig)
plt.close(fig)

print(f"Hybrid winding control points : {hybrid_cfg.winding_ctrl_count}")
print(f"Hybrid patch count            : {hybrid_cfg.patch_count}")
print(f"Hashin margin of safety       : {failure_cfg.margin_of_safety}")
print(f"Friction limit mu_max         : {friction_cfg.mu_max}")
print(f"Baseline strain energy        : {float(np.asarray(baseline['compliance'])):.6f}")
print(f"Baseline mass metric          : {float(np.asarray(baseline['mass_metric'])):.6f}")
print(json.dumps({"allowables": vars(failure_cfg.allowables)}, indent=2))


## Hybrid Optimization

Run the coupled winding + AFP optimizer. The objective is mass-driven and adds sparse patch placement, smooth Hashin penalties, and friction penalties on the winding control field.


In [ ]:
hybrid_run = hostify_tree(
    run_hybrid_optimization(
        state,
        material,
        hybrid_cfg,
        geom,
        solve_compliance,
        failure_config=failure_cfg,
        friction_config=friction_cfg,
    )
)
hybrid_result = hybrid_run["result"]
jax.clear_caches()
gc.collect()

fi_max_with_margin = float(np.max(np.asarray(hybrid_result["failure_with_margin"])))
mu_max_required = float(np.asarray(hybrid_result["mu_max_required"]))

print(json.dumps({
    "history": hybrid_run["history"],
    "fi_max_with_margin": fi_max_with_margin,
    "mu_max_required": mu_max_required,
}, indent=2))


## Objective Influence and Constraint Traces

Show how the added mass, sparsity, Hashin, and friction terms influence the hybrid objective through the continuation schedule.


In [ ]:
history = hybrid_run["history"]
beta = [entry["beta"] for entry in history]
baseline_compliance = float(np.asarray(baseline["compliance"]))
baseline_mass = float(np.asarray(baseline["mass_metric"]))

mass_contrib = [hybrid_cfg.mass_weight * entry["mass_metric"] / baseline_mass for entry in history]
l1_contrib = [hybrid_cfg.patch_l1_weight * entry["patch_l1"] for entry in history]
overlap_contrib = [hybrid_cfg.overlap_penalty_weight * entry["overlap_penalty"] for entry in history]
repulsion_contrib = [hybrid_cfg.repulsion_penalty_weight * entry["repulsion_penalty"] for entry in history]
failure_contrib = [entry["failure_penalty"] for entry in history]
friction_contrib = [entry["friction_penalty"] for entry in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

ax = axes[0]
ax.plot(beta, [entry["objective"] for entry in history], marker="o", color="crimson", label="Objective")
ax.plot(beta, [entry["strain_energy"] / baseline_compliance for entry in history], marker="o", color="steelblue", label="Compliance / baseline")
ax.plot(beta, [entry["mass_metric"] / baseline_mass for entry in history], marker="o", color="darkorange", label="Mass / baseline")
ax.set_xlabel("beta")
ax.set_ylabel("normalized metric")
ax.set_title("Hybrid continuation overview")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=9)

ax = axes[1]
ax.plot(beta, mass_contrib, marker="o", color="darkorange", label="Mass term")
ax.plot(beta, l1_contrib, marker="o", color="slateblue", label="Patch L1 term")
ax.plot(beta, overlap_contrib, marker="o", color="royalblue", label="Overlap term")
ax.plot(beta, repulsion_contrib, marker="o", color="teal", label="Repulsion term")
ax.plot(beta, failure_contrib, marker="o", color="forestgreen", label="Hashin penalty")
ax.plot(beta, friction_contrib, marker="o", color="darkmagenta", label="Friction penalty")
ax.set_xlabel("beta")
ax.set_ylabel("Objective contribution")
ax.set_title("What drives the objective")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

ax = axes[2]
ax.plot(beta, [entry["active_patch_count"] for entry in history], marker="o", color="steelblue", label="Active patch count")
ax.plot(beta, [entry["max_patch_thickness"] for entry in history], marker="o", color="royalblue", label="Max patch thickness")
ax.plot(beta, [entry["max_winding_thickness"] for entry in history], marker="o", color="seagreen", label="Max winding thickness")
ax.plot(beta, [entry["mu_max_required"] for entry in history], marker="o", color="darkmagenta", label="Required friction")
ax.axhline(friction_cfg.mu_max, color="darkmagenta", linestyle="--", linewidth=1.1)
ax.set_xlabel("beta")
ax.set_ylabel("Constraint / design metric")
ax.set_title("Manufacturing and sparsity response")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUTS / "hybrid_objective_influence.png", dpi=120)
display(fig)
plt.close(fig)

final_terms = {
    "objective": history[-1]["objective"],
    "mass_contribution": mass_contrib[-1],
    "patch_l1_contribution": l1_contrib[-1],
    "overlap_contribution": overlap_contrib[-1],
    "repulsion_contribution": repulsion_contrib[-1],
    "failure_penalty": failure_contrib[-1],
    "friction_penalty": friction_contrib[-1],
    "active_patch_count": history[-1]["active_patch_count"],
}
print(json.dumps(final_terms, indent=2))


## Hashin Field and Burst-Pressure Proxy

Map the Hashin result back onto the COPV surface coordinates and use the unit-pressure hybrid stress field to estimate a burst-pressure factor by pressure scaling.


In [ ]:
state_s = np.asarray(jax.device_get(state["s_coords"]))
state_phi_deg = np.degrees(np.asarray(jax.device_get(state["phi_coords"])))

failure_with_margin = np.asarray(hybrid_result["failure_with_margin"])
failure_index = np.asarray(hybrid_result["failure_index"])
fiber_tension = np.asarray(hybrid_result["fiber_tension"])
fiber_compression = np.asarray(hybrid_result["fiber_compression"])
matrix_tension = np.asarray(hybrid_result["matrix_tension"])
matrix_compression = np.asarray(hybrid_result["matrix_compression"])
mode_stack = np.stack([fiber_tension, fiber_compression, matrix_tension, matrix_compression], axis=0)

critical_idx = int(np.argmax(failure_with_margin))
dominant_mode_idx = int(np.argmax(mode_stack[:, critical_idx]))
critical_summary = {
    "critical_element_index": critical_idx,
    "critical_s": float(state_s[critical_idx]),
    "critical_phi_deg": float(state_phi_deg[critical_idx]),
    "critical_failure_index": float(failure_index[critical_idx]),
    "critical_failure_with_margin": float(failure_with_margin[critical_idx]),
    "dominant_mode": FAILURE_MODE_NAMES[dominant_mode_idx],
}

burst_profile = estimate_burst_pressure_profile(
    np.asarray(hybrid_result["local_stress"]),
    failure_cfg.allowables,
    operating_pressure=float(geom.pressure),
    margin_of_safety=float(failure_cfg.margin_of_safety),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
sc = ax.scatter(state_s, state_phi_deg, c=failure_with_margin, s=18, cmap="magma", alpha=0.9)
ax.scatter([critical_summary["critical_s"]], [critical_summary["critical_phi_deg"]], color="cyan", s=64, edgecolor="black")
ax.set_xlabel("Meridional coordinate")
ax.set_ylabel("Azimuth [deg]")
ax.set_title("Hashin failure field on the COPV surface")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Failure index * margin")

ax = axes[1]
for label, values, color in [
    ("Fiber tension", fiber_tension, "crimson"),
    ("Fiber compression", fiber_compression, "royalblue"),
    ("Matrix tension", matrix_tension, "darkorange"),
    ("Matrix compression", matrix_compression, "forestgreen"),
]:
    centers, profile = binned_max_profile(state_s, values)
    ax.plot(centers, profile, linewidth=2, label=label, color=color)
centers, profile = binned_max_profile(state_s, failure_with_margin)
ax.plot(centers, profile, linewidth=2.2, linestyle="--", color="black", label="Max FI * margin")
ax.axhline(1.0, color="0.35", linestyle="--", linewidth=1.1)
ax.set_xlabel("Meridional coordinate")
ax.set_ylabel("Binned max failure metric")
ax.set_title("Where the laminate is critical")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

ax = axes[2]
ax.plot(burst_profile["pressure_factors"], burst_profile["failure_curve"], color="steelblue", linewidth=2, label="Max Hashin FI")
ax.plot(burst_profile["pressure_factors"], burst_profile["failure_curve_with_margin"], color="darkmagenta", linewidth=2, label="Max Hashin FI * margin")
ax.axhline(1.0, color="0.35", linestyle="--", linewidth=1.1)
ax.axvline(burst_profile["burst_factor"], color="steelblue", linestyle=":", linewidth=1.4)
ax.axvline(burst_profile["allowable_factor_with_margin"], color="darkmagenta", linestyle=":", linewidth=1.4)
ax.set_xlabel("Pressure scale factor")
ax.set_ylabel("Failure metric")
ax.set_title("Burst-pressure proxy from the hybrid stress field")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUTS / "hybrid_hashin_burst.png", dpi=120)
display(fig)
plt.close(fig)

burst_summary = {
    **critical_summary,
    "burst_factor": float(burst_profile["burst_factor"]),
    "allowable_factor_with_margin": float(burst_profile["allowable_factor_with_margin"]),
    "burst_pressure": float(burst_profile["burst_pressure"]),
    "allowable_pressure_with_margin": float(burst_profile["allowable_pressure_with_margin"]),
    "mode_at_burst": burst_profile["mode_at_burst"],
}
print(json.dumps(burst_summary, indent=2))


## Manufacturing Visualization and Exports

Visualize the hybrid filament winding path family, inspect the manufacturability field, write a VTU with failure and winding data on the mesh, and export the Abaqus shell deck.


In [ ]:
nodes = mesh.nodes
elems = mesh.elems

base_u = np.asarray(baseline["displacement"]).reshape(len(nodes), 3)
hybrid_u = np.asarray(hybrid_result["displacement"]).reshape(len(nodes), 3)

base_vtu = write_vtu(
    OUTPUTS / "copv_base.vtu",
    nodes,
    elems,
    base_u,
    np.asarray(baseline["thickness"]),
    np.asarray(baseline["density"]),
    np.asarray(baseline["fiber_dirs"]),
    np.asarray(baseline["coverage"]),
)
hybrid_vtu = write_vtu(
    OUTPUTS / "copv_hybrid_jax.vtu",
    nodes,
    elems,
    hybrid_u,
    np.asarray(hybrid_result["thickness"]),
    np.asarray(hybrid_result["density"]),
    np.asarray(hybrid_result["fiber_dirs"]),
    np.asarray(hybrid_result["coverage"]),
    extra_cell_data={
        "failure_index": np.asarray(hybrid_result["failure_index"]),
        "failure_with_margin": np.asarray(hybrid_result["failure_with_margin"]),
        "fiber_tension": np.asarray(hybrid_result["fiber_tension"]),
        "fiber_compression": np.asarray(hybrid_result["fiber_compression"]),
        "matrix_tension": np.asarray(hybrid_result["matrix_tension"]),
        "matrix_compression": np.asarray(hybrid_result["matrix_compression"]),
        "winding_angle_deg": np.degrees(np.asarray(hybrid_result["winding_angle_field"])),
        "winding_added_thickness": np.asarray(hybrid_result["winding_thickness_field"]),
        "patch_added_thickness": np.asarray(hybrid_result["patch_added_thickness"]),
    },
)

hybrid_patch_state = {
    "s_coords": np.asarray(hybrid_result["patch_s"]),
    "phis": np.asarray(hybrid_result["patch_phi"]),
    "alphas": np.asarray(hybrid_result["patch_alpha"]),
}
hybrid_patch_layout = None
hybrid_patch_layout_path = None
if hybrid_cfg.patch_count > 0:
    hybrid_patch_layout = build_patch_layout_data(hybrid_patch_state, verification_patch_cfg, geom)
    hybrid_patch_layout_path = save_layout_json(OUTPUTS / "hybrid_patch_layout.json", hybrid_patch_layout)

    fig = plot_patch_projection(
        hybrid_patch_state,
        verification_patch_cfg,
        geom,
        OUTPUTS / "copv_hybrid_patch_projection.png",
    )
    display(fig)
    plt.close(fig)

fig, hybrid_winding_layout = plot_hybrid_winding_paths(
    hybrid_result,
    geom,
    family_count=8,
    sample_count=280,
    save_path=OUTPUTS / "copv_hybrid_winding_paths.png",
)
display(fig)
plt.close(fig)

hybrid_winding_layout = build_hybrid_winding_layout_data(
    hybrid_result,
    geom,
    family_count=8,
    sample_count=280,
)
hybrid_winding_layout_path = save_layout_json(OUTPUTS / "hybrid_winding_layout.json", hybrid_winding_layout)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

ax = axes[0]
ax.plot(hybrid_winding_layout["sample_s"], hybrid_winding_layout["angle_profile_deg"], color="seagreen", linewidth=2, label="Angle profile")
ax.scatter(hybrid_winding_layout["control_s"], hybrid_winding_layout["control_angle_deg"], color="black", s=28, zorder=3, label="Control points")
ax.set_xlabel("Meridional coordinate")
ax.set_ylabel("Winding angle [deg]")
ax.set_title("Hybrid winding angle field")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=9)

ax = axes[1]
ax.plot(hybrid_winding_layout["sample_s"], hybrid_winding_layout["thickness_profile"], color="royalblue", linewidth=2, label="Winding thickness field")
if hybrid_cfg.patch_count > 0:
    ax.scatter(
        np.asarray(hybrid_result["patch_s"]),
        np.asarray(hybrid_result["patch_thickness"]),
        color="steelblue",
        s=34,
        label="AFP patch thickness",
    )
ax.set_xlabel("Meridional coordinate / patch center")
ax.set_ylabel("Added thickness")
ax.set_title("How winding and AFP share material")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

ax = axes[2]
ax.plot(hybrid_winding_layout["sample_s"], hybrid_winding_layout["mu_required"], color="darkmagenta", linewidth=2, label="Required friction")
ax.axhline(friction_cfg.mu_max, color="0.30", linestyle="--", linewidth=1.1, label="Allowable friction")
ax.set_xlabel("Meridional coordinate")
ax.set_ylabel("Required friction coefficient")
ax.set_title("Manufacturability constraint on the winding field")
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUTS / "hybrid_manufacturing_constraints.png", dpi=120)
display(fig)
plt.close(fig)

hybrid_explicit_screenshot = None
try:
    hybrid_explicit_screenshot = save_explicit_manufacturing_layout_screenshot(
        base_vtu,
        OUTPUTS / "pyvista_hybrid_explicit_layout.png",
        curve_points_list=[entry["points"] for entry in hybrid_winding_layout["paths"]],
        curve_colors=[
            "forestgreen" if entry["handedness"] == "clockwise" else "darkmagenta"
            for entry in hybrid_winding_layout["paths"]
        ],
        patch_polygons=None if hybrid_patch_layout is None else [entry["corners"] for entry in hybrid_patch_layout["patches"]],
        tow_radius=1.15,
        title="Hybrid winding + AFP layout over COPV",
    )
    display(Image(filename=str(hybrid_explicit_screenshot)))
except Exception as exc:
    print(f"Skipped off-screen PyVista manufacturing screenshot: {exc}")

abaqus_path = export_result_to_abaqus(
    state,
    hybrid_result,
    geom,
    OUTPUTS / "optimized_copv_hybrid.inp",
    material=material,
    heading="Hybrid COPV layup exported from the main JAX workflow notebook",
)

fi_max = float(np.asarray(hybrid_result["fi_max"]))
fi_max_with_margin = float(np.max(np.asarray(hybrid_result["failure_with_margin"])))
mu_max_required = float(np.asarray(hybrid_result["mu_max_required"]))
patch_thickness = np.asarray(hybrid_result["patch_thickness"])
active_patch_count = int(np.sum(patch_thickness > 0.05 * hybrid_cfg.max_patch_thickness))

summary = {
    "jax_backend": jax.default_backend(),
    "mesh": {
        "nodes": int(len(nodes)),
        "elements": int(len(elems)),
        "step": str(step_path),
        "msh": str(msh_path),
        "mesh_hmin": float(geom.mesh_hmin),
        "mesh_hmax": float(geom.mesh_hmax),
    },
    "failure_config": {
        "allowables": vars(failure_cfg.allowables),
        "margin_of_safety": float(failure_cfg.margin_of_safety),
        "penalty_weight": float(failure_cfg.penalty_weight),
    },
    "friction_config": {
        "mu_max": float(friction_cfg.mu_max),
        "penalty_weight": float(friction_cfg.penalty_weight),
    },
    "hybrid_config": {
        "winding_ctrl_count": int(hybrid_cfg.winding_ctrl_count),
        "patch_count": int(hybrid_cfg.patch_count),
        "beta_schedule": [float(x) for x in hybrid_cfg.beta_schedule],
        "patch_l1_weight": float(hybrid_cfg.patch_l1_weight),
        "max_winding_thickness": float(hybrid_cfg.max_winding_thickness),
        "max_patch_thickness": float(hybrid_cfg.max_patch_thickness),
    },
    "baseline": {
        "strain_energy": float(np.asarray(baseline["compliance"])),
        "mass_metric": float(np.asarray(baseline["mass_metric"])),
        "vtu": str(base_vtu),
    },
    "hybrid": {
        "history": hybrid_run["history"],
        "objective": float(np.asarray(hybrid_result["objective"])),
        "strain_energy": float(np.asarray(hybrid_result["compliance"])),
        "mass_metric": float(np.asarray(hybrid_result["mass_metric"])),
        "strain_energy_ratio_vs_baseline": float(np.asarray(hybrid_result["compliance"])) / float(np.asarray(baseline["compliance"])),
        "mass_ratio_vs_baseline": float(np.asarray(hybrid_result["mass_metric"])) / float(np.asarray(baseline["mass_metric"])),
        "fi_max": fi_max,
        "fi_max_with_margin": fi_max_with_margin,
        "friction_mu_max_required": mu_max_required,
        "friction_mu_allowable": float(friction_cfg.mu_max),
        "hashin_constraint_satisfied": bool(fi_max_with_margin <= 1.0 + 1e-6),
        "friction_constraint_satisfied": bool(mu_max_required <= friction_cfg.mu_max + 1e-6),
        "burst_factor": float(burst_profile["burst_factor"]),
        "allowable_factor_with_margin": float(burst_profile["allowable_factor_with_margin"]),
        "allowable_pressure_with_margin": float(burst_profile["allowable_pressure_with_margin"]),
        "dominant_failure_mode": critical_summary["dominant_mode"],
        "critical_surface_coordinate": {
            "s": critical_summary["critical_s"],
            "phi_deg": critical_summary["critical_phi_deg"],
        },
        "active_patch_count": active_patch_count,
        "vtu": str(hybrid_vtu),
        "abaqus_inp": str(abaqus_path),
    },
    "visualisations": {
        "objective_influence_plot": str(OUTPUTS / "hybrid_objective_influence.png"),
        "hashin_burst_plot": str(OUTPUTS / "hybrid_hashin_burst.png"),
        "winding_plot": str(OUTPUTS / "copv_hybrid_winding_paths.png"),
        "manufacturing_constraints_plot": str(OUTPUTS / "hybrid_manufacturing_constraints.png"),
        "patch_projection": str(OUTPUTS / "copv_hybrid_patch_projection.png") if hybrid_patch_layout is not None else None,
        "patch_layout_json": str(hybrid_patch_layout_path) if hybrid_patch_layout_path is not None else None,
        "winding_layout_json": str(hybrid_winding_layout_path),
        "pyvista_hybrid_explicit": None if hybrid_explicit_screenshot is None else str(hybrid_explicit_screenshot),
    },
}

summary_path = OUTPUTS / "hybrid_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

preview = "\n".join(abaqus_path.read_text(encoding="utf-8").splitlines()[:20])
print(preview)
print(json.dumps(summary, indent=2))


## Interactive Viewer Hooks

Keep these disabled in the committed notebook, but they are convenient when running locally with a GUI-enabled environment.


In [ ]:
# Uncomment locally for interactive inspection:
#
# render_vtu_interactive(OUTPUTS / "copv_hybrid_jax.vtu", scalar_field="failure_with_margin", slice_model=True)
# if hybrid_patch_layout is not None:
#     render_explicit_manufacturing_layout(
#         OUTPUTS / "copv_base.vtu",
#         curve_points_list=[entry["points"] for entry in hybrid_winding_layout["paths"]],
#         patch_polygons=[entry["corners"] for entry in hybrid_patch_layout["patches"]],
#         title="Hybrid winding + AFP layout over COPV",
#     )
